In [1]:
import logging

from dotenv import load_dotenv

from utils.data_helpers import initialize_metadata_data, initialize_stock_data

load_dotenv()

logger = logging.getLogger(__name__)


await initialize_stock_data()
await initialize_metadata_data()

2026-02-18 21:10:02,844 - utils.data_helpers - INFO - Initializing stock data mappings...
2026-02-18 21:10:03,193 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/findActiveEquityAndSublisting "HTTP/1.1 200 OK"
2026-02-18 21:10:03,413 - utils.data_helpers - INFO - Stock data initialized: 5525 stocks, 5509 symbols
2026-02-18 21:10:03,415 - utils.data_helpers - INFO - Initializing metadata mappings (20 years of data)...
2026-02-18 21:10:04,752 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getMetaDataRelatedToFile?startDate=2006-02-23&endDate=2026-02-18 "HTTP/1.1 200 OK"
2026-02-18 21:10:04,988 - utils.data_helpers - INFO - Metadata initialized: 22109 items, 4064 unique fincodes


In [2]:
from rag.ingestion.document_fetcher import DocumentFetcher

fetcher = DocumentFetcher()

d:\work\finSharpe\repos\embed-documents-main\.venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [7]:
test_fincode = 100325  # Bajaj Finance Ltd.
docs = await fetcher.get_available_documents(
    fincode=test_fincode,
)
for doc in docs:
    logger.info(f"Document: {doc.filename} ({doc.category}) - {doc.document_date}")

2026-02-18 21:31:47,308 - rag.ingestion.document_fetcher - INFO - Fetching document metadata: fincode=100325, category=None, date_range=2024-02-19 to 2026-02-18
2026-02-18 21:31:47,323 - utils.api_utils - INFO - Retrieved meta data file for 2024-02-19 to 2026-02-18 from cache
2026-02-18 21:31:47,325 - rag.ingestion.document_fetcher - INFO - Found 13 documents matching filters
2026-02-18 21:31:47,325 - __main__ - INFO - Document: f2454c0f-aad9-4d23-add0-dfe3d49aa8d6.pdf (concall) - 2025-10-19
2026-02-18 21:31:47,326 - __main__ - INFO - Document: b74bb5fb-c15b-42f1-b853-34231e587f46.pdf (investor-presentation) - 2025-10-17
2026-02-18 21:31:47,326 - __main__ - INFO - Document: 7a050d55-ad53-4d57-8735-cbcf20bb412b.pdf (investor-presentation) - 2025-10-17
2026-02-18 21:31:47,326 - __main__ - INFO - Document: 38a33049-ed9e-4544-ace5-a92e5f984f23.pdf (concall) - 2025-07-20
2026-02-18 21:31:47,327 - __main__ - INFO - Document: 49a01d99-854c-4c3a-b62c-e1fb3619029f.pdf (investor-presentation) - 

In [8]:
from utils.data_helpers import fincode_to_symbol

company_name = fincode_to_symbol(test_fincode)

In [9]:
company_name

'RELIANCE'

In [10]:
for doc in docs:
    try:
        await fetcher.save_document_to_disk(
            doc.filename, doc.category, f".cache/company_documents/{company_name}"
        )
    except Exception as e:
        logger.error(f"Error saving document {doc.filename} to disk: {e}")
        continue

2026-02-18 21:31:58,258 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=concall&attachmentName=f2454c0f-aad9-4d23-add0-dfe3d49aa8d6.pdf "HTTP/1.1 200 OK"
2026-02-18 21:31:58,359 - rag.ingestion.document_fetcher - INFO - Saved PDF to disk: .cache/company_documents/RELIANCE\f2454c0f-aad9-4d23-add0-dfe3d49aa8d6.pdf
2026-02-18 21:31:58,900 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=investor-presentation&attachmentName=b74bb5fb-c15b-42f1-b853-34231e587f46.pdf "HTTP/1.1 200 OK"
2026-02-18 21:31:59,824 - rag.ingestion.document_fetcher - INFO - Saved PDF to disk: .cache/company_documents/RELIANCE\b74bb5fb-c15b-42f1-b853-34231e587f46.pdf
2026-02-18 21:32:00,399 - httpx - INFO - HTTP Request: GET https://radar.definedgesecurities.com/v1/data-feed/getFileByCategoryAndAttachmentName?subcatname=investor-presentation&attachmentNa